# 1. 무엇을 배우는 실험인가?

직전 행동을 구분하면 **기록된 처리 시도 간격**을 더 잘 설명할까요?
B0는 항상 Train 평균, B1은 Train 전체 분포, B2는 이전 행동별 Train 분포입니다.
신경망·다음 행동·배치·Hold를 학습하는 실험이 아닙니다. 실제 키 누름·반응시간으로 해석하지 않습니다.

Train으로만 분포를 만들고 Validation으로 비교합니다. Test 파일은 열지 않습니다.
기존 0ms와 긴 간격을 보존합니다. Smoke는 계산 확인용이며 모델 성능의 근거가 아닙니다.


# 2. 준비와 재사용

기존 readiness Notebook의 checksum·JSON·경로 보호·독점 게시 함수를 정의 셀에서 재사용합니다.
원본 로그 parser를 실행하거나 dataset을 다시 만들지 않습니다. 아래 정의 셀 자체는 fit을 실행하지 않습니다.


In [ ]:
import hashlib
import json
import math
import os
import platform
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import numpy as np

EXPERIMENT_ID = "human-timing-baseline"
ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_timing_baseline.ipynb"
DEPENDENCY = ROOT / "scripts/ml/notebooks/human_action_data_readiness.ipynb"
shared = {"__name__": "timing_baseline_shared"}
for cell in json.loads(DEPENDENCY.read_text(encoding="utf-8"))["cells"]:
    if cell["cell_type"] == "code" and "definitions" in cell["metadata"].get("tags", []):
        exec(compile("".join(cell["source"]), str(DEPENDENCY), "exec"), shared)
shared["EXPERIMENT_ID"] = EXPERIMENT_ID
for name in ("require", "finite", "integer", "json_bytes", "sha256_file",
             "iter_rows", "reject_constant", "checked_paths", "publish_report"):
    globals()[name] = shared[name]

DEFAULT_DATASET = "data/datasets/human-timing-v1-20261024-full"
EXPECTED_MANIFEST_SHA = "c203a1db7b8f2aa7014c565b371088a9614f6abb0df5632e93fe6c807712fc88"
ACTIONS = shared["ACTIONS"] - {"hold"}
MODEL_IDS = ("B0", "B1", "B2")
TARGET_GAME_ID = "119e01c9-b982-4907-9480-2308f4604635"


# 3. 고정 입력 확인하기

Manifest SHA와 Train/Validation의 bytes·SHA·행 수를 확인합니다.
각 pair의 ID·인접 index·game split을 검사합니다. Smoke에서도
입력 파일 검증과 모델에 전달하는 subset을 구분합니다. Test는 manifest의 game 목록만 확인하고 파일은 읽지 않습니다.


In [ ]:
def validate_sample(row, split, games):
    """정제 JSONL의 schema와 인접 lineage를 확인하고 label을 보정하지 않는다."""
    fields = {"schemaVersion", "sampleId", "gameId", "playerNumber", "pieceIndex",
              "sourceLine", "previousInputIndex", "currentInputIndex", "previousAction",
              "nextActionDiagnostic", "recordedIntervalMs", "split"}
    require(isinstance(row, dict) and set(row) == fields, "SAMPLE_SCHEMA_INVALID")
    require(type(row["schemaVersion"]) is int and row["schemaVersion"] == 1, "SAMPLE_VERSION_INVALID")
    require(row["split"] == split and isinstance(row["gameId"], str)
            and row["gameId"] in games, "SAMPLE_SPLIT_INVALID")
    require(all(integer(row[k]) for k in ("playerNumber", "pieceIndex", "sourceLine",
            "previousInputIndex", "currentInputIndex"))
            and row["playerNumber"] > 0 and row["sourceLine"] > 0
            and row["currentInputIndex"] == row["previousInputIndex"] + 1, "PAIR_LINEAGE_INVALID")
    require(isinstance(row["previousAction"], str) and row["previousAction"] in ACTIONS
            and isinstance(row["nextActionDiagnostic"], str)
            and row["nextActionDiagnostic"] in ACTIONS, "ACTION_INVALID")
    require(finite(row["recordedIntervalMs"]) and row["recordedIntervalMs"] >= 0, "INTERVAL_INVALID")
    lineage = {k: row[k] for k in ("gameId", "playerNumber", "pieceIndex", "sourceLine",
                                  "previousInputIndex", "currentInputIndex")}
    require(row["sampleId"] == hashlib.sha256(json_bytes(lineage)).hexdigest(), "SAMPLE_ID_INVALID")


def load_inputs(dataset_name=DEFAULT_DATASET, mode="smoke"):
    """고정 manifest와 두 입력 파일을 검증한 뒤 smoke/full Train·Validation을 반환한다."""
    require(mode in ("smoke", "full"), "INVALID_RUN_MODE")
    relative = Path(dataset_name)
    require(not relative.is_absolute() and ".." not in relative.parts, "DATASET_MUST_BE_REPO_RELATIVE")
    directory = (ROOT / relative).resolve(strict=True)
    require(directory.is_relative_to(ROOT) and directory.is_dir(), "DATASET_OUTSIDE_REPOSITORY")
    manifest_path = directory / "manifest.json"
    require(sha256_file(manifest_path) == EXPECTED_MANIFEST_SHA, "MANIFEST_SHA_MISMATCH")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"), parse_constant=reject_constant)
    require(manifest.get("datasetSchemaVersion") == 1
            and manifest.get("experimentId") == "human-timing-dataset"
            and manifest.get("runMode") == "full" and manifest.get("fullCohortCovered") is True
            and manifest.get("dataDecisionEligible") is True
            and manifest.get("featureColumns") == ["previousAction"]
            and manifest.get("labelColumn") == "recordedIntervalMs", "MANIFEST_CONTRACT_INVALID")
    groups = manifest["split"]["gameIds"]
    require(set(groups) == {"train", "validation", "test"}
            and all(isinstance(g, list) and g and all(isinstance(x, str) and x for x in g)
                    and g == sorted(set(g)) for g in groups.values()), "GAME_LIST_INVALID")
    require(not any(set(groups[a]) & set(groups[b]) for a, b in
                    (("train", "validation"), ("train", "test"), ("validation", "test"))),
            "GAME_SPLIT_LEAKAGE")
    require(sorted(sum(groups.values(), [])) == manifest["cohort"]["selectedGameIds"], "COHORT_MISMATCH")
    watched = {manifest_path: EXPECTED_MANIFEST_SHA}
    rows, seen = {}, set()
    for split, limit in (("train", 5), ("validation", 3)):
        path = directory / (split + ".jsonl")
        require(path.resolve(strict=True).is_relative_to(directory), "INPUT_OUTSIDE_DATASET")
        info = manifest["files"][path.name]
        require(path.stat().st_size == info["bytes"] and sha256_file(path) == info["sha256"],
                "DATASET_FILE_SHA_MISMATCH")
        watched[path] = info["sha256"]
        all_rows = []
        for _, row, kind in iter_rows(path):
            require(kind == "parsed", "DATASET_ROW_INVALID")
            validate_sample(row, split, set(groups[split]))
            require(row["sampleId"] not in seen, "DUPLICATE_SAMPLE")
            seen.add(row["sampleId"])
            all_rows.append(row)
        require(len(all_rows) == info["rows"] and {r["gameId"] for r in all_rows} == set(groups[split]),
                "SPLIT_COUNT_MISMATCH")
        selected = set(groups[split][:limit] if mode == "smoke" else groups[split])
        rows[split] = sorted((r for r in all_rows if r["gameId"] in selected), key=lambda r: r["sampleId"])
    require(all(sha256_file(p) == h for p, h in watched.items()), "INPUT_CHANGED")
    return manifest, rows, watched


# 4. 작은 예제로 분포 점수 이해하기

Train 간격이 [0, 100]ms이고 관측이 0ms라면, 전체 분포 CRPS는 25ms입니다.
고정 평균 50ms의 점수는 50ms입니다. 점수가 작을수록 좋습니다.
임의로 한 번 샘플링한 오차가 아니라 분포 전체를 비교합니다.
정렬·누적합으로 계산하므로 큰 거리 행렬을 만들지 않습니다.


In [ ]:
def prepare_distribution(values):
    """같은 확률을 갖는 경험 표본을 정렬하고 CRPS 재사용 항을 계산한다."""
    require(isinstance(values, (list, tuple)) and len(values) > 0
            and all(finite(x) and x >= 0 for x in values), "SUPPORT_INVALID")
    support = np.sort(np.asarray(values, dtype=float))
    n = len(support)
    prefix = np.concatenate(([0.0], np.cumsum(support)))
    half_distance = math.fsum((2 * i - n + 1) * float(x) for i, x in enumerate(support)) / n**2
    require(np.isfinite(prefix).all() and finite(half_distance), "SUPPORT_OVERFLOW")
    return support, prefix, half_distance


def crps(prepared, observations):
    """예측 분포를 바꾸지 않고 각 관측의 CRPS를 ms 단위로 반환한다."""
    require(len(observations) > 0 and all(finite(y) and y >= 0 for y in observations), "OBSERVATION_INVALID")
    x, prefix, half_distance = prepared
    y = np.asarray(observations, dtype=float)
    k = np.searchsorted(x, y, side="right")
    result = (y * k - prefix[k] + prefix[-1] - prefix[k] - y * (len(x) - k)) / len(x) - half_distance
    require(np.isfinite(result).all() and (result >= -1e-10).all(), "CRPS_INVALID")
    return np.maximum(result, 0)


def distribution_summary(values, weights=None):
    x = np.asarray(values, dtype=float)
    w = np.ones(len(x), dtype=float) if weights is None else np.asarray(weights, dtype=float)
    require(x.ndim == 1 and len(x) > 0 and w.shape == x.shape
            and np.isfinite(x).all() and (x >= 0).all()
            and np.isfinite(w).all() and (w > 0).all() and finite(float(w.sum())), "DISTRIBUTION_INVALID")
    order = np.argsort(x, kind="stable")
    x, w = x[order], w[order] / w.sum()
    cumulative = np.cumsum(w)
    uniform = bool(np.all(w == w[0]))
    def quantile(q):
        index = math.ceil(q * len(x)) - 1 if uniform else int(np.searchsorted(cumulative, q, side="left"))
        index = min(index, len(x) - 1)
        return float(x[index])
    return {"meanMs": float(np.dot(x, w)), "medianMs": quantile(.5), "p95Ms": quantile(.95),
            "zeroProbability": float(w[x == 0].sum())}


In [ ]:
example = prepare_distribution([0, 100])
assert crps(example, [0]).tolist() == [25.0]
assert crps(prepare_distribution([50]), [0]).tolist() == [50.0]
print("Synthetic example: empirical CRPS=25ms, fixed mean CRPS=50ms; not a real result.")


# 5. Train만으로 세 baseline 만들기

B0/B1은 전체 Train을 사용합니다. B2는 previousAction으로만 묶습니다.
다음 행동·승패·게임 ID는 예측 feature가 아닙니다. Game ID는 누수 검사와 평가 집계에만 씁니다.
분포를 fit하는 함수에는 Validation을 전달하지 않습니다. 학습되지 않은 정상 행동은 B1로 fallback합니다.


In [ ]:
def fit_baselines(train):
    """Train-only empirical candidate bundle. 신경망이나 production 모델이 아니다."""
    require(train and all(r.get("split") == "train" for r in train), "TRAIN_ONLY_REQUIRED")
    require(all(isinstance(r.get("previousAction"), str) and r["previousAction"] in ACTIONS
                and finite(r.get("recordedIntervalMs")) and r["recordedIntervalMs"] >= 0 for r in train),
            "TRAIN_SAMPLE_INVALID")
    values = sorted(r["recordedIntervalMs"] for r in train)
    prepare_distribution(values)
    conditional = {a: sorted(r["recordedIntervalMs"] for r in train if r["previousAction"] == a)
                   for a in sorted({r["previousAction"] for r in train})}
    return {"bundleSchemaVersion": 1, "feature": "previousAction", "label": "recordedIntervalMs",
            "trainGameIds": sorted({r["gameId"] for r in train}), "trainPairs": len(train),
            "fitCount": 3, "models": {"B0": [math.fsum(values) / len(values)], "B1": values, "B2": conditional},
            "canonicalArtifact": False, "productionIntegration": False}


def forecast_values(bundle, model_id, action_counts):
    """평가 입력의 action 비율로 Train 분포를 혼합한다. 평가 label은 사용하지 않는다."""
    require(model_id in MODEL_IDS and action_counts and all(a in ACTIONS and type(n) is int and n > 0
            for a, n in action_counts.items()), "FORECAST_INPUT_INVALID")
    models = bundle["models"]
    if model_id != "B2":
        return models[model_id], [1.0] * len(models[model_id])
    values, weights = [], []
    for action, count in sorted(action_counts.items()):
        support = models["B2"].get(action, models["B1"])
        values.extend(support)
        weights.extend([count / len(support)] * len(support))
    return values, weights


def evaluate_bundle(bundle, rows):
    """동일 bundle을 읽기 전용으로 평가하며 어떤 분포도 다시 fit하지 않는다."""
    require(rows and len({r["split"] for r in rows}) == 1
            and rows[0]["split"] in ("validation", "test"), "EVALUATION_SPLIT_INVALID")
    require(not set(bundle["trainGameIds"]) & {r["gameId"] for r in rows}, "EVALUATION_GAME_LEAKAGE")
    require(all(isinstance(r.get("previousAction"), str) and r["previousAction"] in ACTIONS
                and finite(r.get("recordedIntervalMs")) and r["recordedIntervalMs"] >= 0 for r in rows),
            "EVALUATION_SAMPLE_INVALID")
    before = json_bytes(bundle)
    games = sorted({r["gameId"] for r in rows})
    actions = Counter(r["previousAction"] for r in rows)
    observed = [r["recordedIntervalMs"] for r in rows]
    metrics, per_game, per_action = [], [], []
    for model_id in MODEL_IDS:
        scores, absolute = np.empty(len(rows)), np.empty(len(rows))
        fallback = 0
        for action in sorted(actions):
            indices = [i for i, r in enumerate(rows) if r["previousAction"] == action]
            support = (bundle["models"]["B2"].get(action, bundle["models"]["B1"])
                       if model_id == "B2" else bundle["models"][model_id])
            forecast = prepare_distribution(support)
            labels = [observed[i] for i in indices]
            scores[indices] = crps(forecast, labels)
            summary = distribution_summary(support)
            absolute[indices] = np.abs(np.asarray(labels) - summary["medianMs"])
            fallback += len(indices) if model_id == "B2" and action not in bundle["models"]["B2"] else 0
            per_action.append({"model": model_id, "action": action, "pairs": len(indices),
                               "crpsMs": float(np.mean(scores[indices])), "predicted": summary,
                               "observed": distribution_summary(labels)})
        game_scores = []
        for game in games:
            indices = [i for i, r in enumerate(rows) if r["gameId"] == game]
            value = float(np.mean(scores[indices]))
            game_scores.append(value)
            per_game.append({"model": model_id, "gameId": game, "pairs": len(indices),
                             "crpsMs": value, "medianMaeMs": float(np.mean(absolute[indices]))})
        values, weights = forecast_values(bundle, model_id, actions)
        metrics.append({"model": model_id, "gameMacroCrpsMs": math.fsum(game_scores) / len(games),
                        "pairMicroCrpsMs": float(np.mean(scores)), "medianMaeMs": float(np.mean(absolute)),
                        "unseenActionFallbackPairs": fallback, "predicted": distribution_summary(values, weights)})
    by_game = {(r["model"], r["gameId"]): r["crpsMs"] for r in per_game}
    comparison = [{"gameId": g, "deltaB2MinusB1Ms": by_game["B2", g] - by_game["B1", g]} for g in games]
    delta = metrics[2]["gameMacroCrpsMs"] - metrics[1]["gameMacroCrpsMs"]
    require(before == json_bytes(bundle), "BUNDLE_MUTATED")
    return {"games": len(games), "pairs": len(rows), "actionCounts": dict(sorted(actions.items())),
            "metrics": metrics, "perGame": per_game, "perAction": per_action,
            "observed": distribution_summary(observed), "gameComparison": comparison,
            "improvedGames": sum(r["deltaB2MinusB1Ms"] < -1e-12 for r in comparison),
            "tiedGames": sum(abs(r["deltaB2MinusB1Ms"]) <= 1e-12 for r in comparison),
            "worsenedGames": sum(r["deltaB2MinusB1Ms"] > 1e-12 for r in comparison),
            "deltaB2MinusB1Ms": delta,
            "comparison": "tie" if abs(delta) <= 1e-12 else ("B2_lower" if delta < 0 else "B1_lower"),
            "evaluationFitCount": 0}


def sensitivity_views(bundle, rows, flagged):
    """후속 승인 Test 평가의 mask 계약을 synthetic 입력으로 검사하는 함수다."""
    require(rows and all(r["split"] == "test" for r in rows), "TEST_ROWS_REQUIRED")
    selected = [r for r in rows if r["gameId"] == flagged]
    other = [r for r in rows if r["gameId"] != flagged]
    require(selected and other, "SENSITIVITY_GROUP_EMPTY")
    digest = hashlib.sha256(json_bytes(bundle)).hexdigest()
    return {name: {"bundleSha256": digest, **evaluate_bundle(bundle, subset)}
            for name, subset in (("test_all", rows), ("test_without_flagged_game", other),
                                  ("flagged_game_only", selected))}


# 6. 작은 실행에서 전체 Validation으로

기본은 smoke: manifest 정렬 순서의 Train 앞 5게임과 Validation 앞 3게임입니다.
Full은 Train 149게임·Validation 31게임이며 **Test 평가가 아닙니다**.
환경변수 HUMAN_TIMING_BASELINE_MODE, HUMAN_TIMING_BASELINE_DATASET, HUMAN_TIMING_BASELINE_REPORT를 사용합니다.
Full 설정 자체는 실행 승인이 아닙니다. 기존 report는 덮어쓰지 않으므로 새 경로가 필요합니다.


In [ ]:
def run_baseline_from_environment():
    """검증→Train fit→Validation score→불변성 확인→독점 report 게시."""
    mode = os.environ.get("HUMAN_TIMING_BASELINE_MODE", "smoke")
    dataset = os.environ.get("HUMAN_TIMING_BASELINE_DATASET", DEFAULT_DATASET)
    require(mode != "full" or "HUMAN_TIMING_BASELINE_REPORT" in os.environ, "FULL_REPORT_PATH_REQUIRED")
    output_name = os.environ.get("HUMAN_TIMING_BASELINE_REPORT", "/private/tmp/human-timing-baseline-smoke.json")
    output_name = str(ROOT / output_name) if not Path(output_name).is_absolute() else output_name
    _, output = checked_paths(ROOT, str(Path(dataset) / "manifest.json"), output_name, mode)
    manifest, rows, watched = load_inputs(dataset, mode)
    watched.update({SOURCE_NOTEBOOK: sha256_file(SOURCE_NOTEBOOK), DEPENDENCY: sha256_file(DEPENDENCY)})
    bundle = fit_baselines(rows["train"])
    bundle["provenance"] = {"manifestSha256": EXPECTED_MANIFEST_SHA,
                            "trainSha256": manifest["files"]["train.jsonl"]["sha256"],
                            "sourceNotebookSha256": watched[SOURCE_NOTEBOOK],
                            "dependencyNotebookSha256": watched[DEPENDENCY]}
    bundle_sha = hashlib.sha256(json_bytes(bundle)).hexdigest()
    result = evaluate_bundle(bundle, rows["validation"])
    train_summary = [{"action": a, "pairs": len(values), **distribution_summary(values)}
                     for a, values in bundle["models"]["B2"].items()]
    require(all(sha256_file(p) == h for p, h in watched.items()), "INPUT_CHANGED")
    report = {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "runMode": mode,
              "status": "timing_baseline_smoke_valid" if mode == "smoke" else "validation_baseline_comparison_complete",
              "fullCohortCovered": mode == "full", "policyDecisionEligible": mode == "full",
              "canonicalResultEligible": False, "productionIntegration": False,
              "candidateBundle": bundle, "candidateBundleSha256": bundle_sha,
              "dataset": {"path": dataset, "manifestSha256": EXPECTED_MANIFEST_SHA,
                          "files": {s: manifest["files"][s + ".jsonl"] for s in ("train", "validation")}},
              "cohort": {s: {"gameIds": sorted({r["gameId"] for r in samples}), "pairs": len(samples)}
                         for s, samples in rows.items()},
              "primary": "validation_game_macro_crps_ms_lower_is_better",
              "validation": result, "trainActionSummary": train_summary,
              "trainActionsNotEvaluated": sorted(set(bundle["models"]["B2"]) - set(result["actionCounts"])),
              "inputUnchanged": True, "testFileRead": False, "testEvaluated": False, "gameIdLeakage": 0,
              "modelFitCount": 3, "candidateSelection": "needs_policy_decision" if mode == "full" else "not_eligible",
              "environment": {"python": platform.python_version(),
                              **{name: version(name) for name in ("numpy", "pandas", "matplotlib")}},
              "limits": ["recorded_processing_attempt_interval_not_keypress_or_reaction_time",
                         "no_state_action_hold_or_personalization_model",
                         "pair_weighted_train_game_macro_evaluation",
                         "historical_zero_ms_origin_unverified", "no_gameplay_or_production_evidence"]}
    json_bytes(report)
    publish_report(report, output)
    return report


In [ ]:
REPORT = run_baseline_from_environment()
print(json.dumps({k: REPORT[k] for k in ("experimentId", "runMode", "status", "testFileRead",
                                        "candidateBundleSha256")}, ensure_ascii=False, indent=2))


# 7. 표와 그래프 읽기

아래 표는 실제 사용한 subset의 수치입니다. gameMacroCrpsMs가 primary이며 작을수록 좋습니다.
게임별 B2−B1이 음수면 그 게임에서는 B2의 분포 점수가 낮습니다. 통계적 우월·체감 개선은 뜻하지 않습니다.
CDF는 x ms 이하 간격의 비율, 0ms 막대는 관측/예측 비율입니다. 원자료는 자르지 않습니다.
Report에 candidateBundle이 함께 있으므로 모델·결과 파일을 별도로 복제하지 않아도 같은 계산을 추적할 수 있습니다.


In [ ]:
def show_results(report):
    """보존한 report만 표시한다. raw 재추출·fit·score는 실행하지 않는다."""
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display

    result, bundle = report["validation"], report["candidateBundle"]
    print(f"Mode: {report['runMode']} | Validation {result['games']} games / {result['pairs']:,} pairs")
    print("Train summaries: counts are pairs; times are milliseconds.")
    display(pd.DataFrame(report["trainActionSummary"]).set_index("action"))
    display(pd.json_normalize(result["metrics"]).set_index("model"))
    display(pd.DataFrame([result["observed"]], index=["Observed Validation"]))
    display(pd.DataFrame(result["perGame"]).sort_values(["model", "crpsMs"], ascending=[True, False]))
    display(pd.json_normalize(result["perAction"]))
    print("Train actions not evaluated:", report["trainActionsNotEvaluated"])

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
    for model_id in MODEL_IDS:
        values, weights = forecast_values(bundle, model_id, result["actionCounts"])
        x, w = np.asarray(values), np.asarray(weights)
        order = np.argsort(x, kind="stable")
        plot_x = np.concatenate(([0.0], x[order], [max(bundle["models"]["B1"])]))
        plot_y = np.concatenate(([0.0], np.cumsum(w[order]) / w.sum(), [1.0]))
        axes[0].step(plot_x, plot_y, where="post", label=model_id)
    axes[0].set(title="Train forecast CDF at Validation action mix", xlabel="Recorded interval (ms)",
                ylabel="Cumulative probability", ylim=(0, 1.02))
    axes[0].legend()
    zero = pd.DataFrame([{"model": r["model"], "probability": r["predicted"]["zeroProbability"]}
                         for r in result["metrics"]] +
                        [{"model": "Observed Validation", "probability": result["observed"]["zeroProbability"]}])
    zero.set_index("model").plot.bar(ax=axes[1], legend=False, color="#0072B2", rot=20)
    axes[1].set(title="Zero-ms probability", ylabel="Fraction of pairs", xlabel="")
    fig.suptitle(f"Timing baseline | {report['runMode']} | not gameplay evidence")
    plt.show()
    plt.close(fig)

    comparison = pd.DataFrame(result["gameComparison"]).set_index("gameId")
    fig, ax = plt.subplots(figsize=(10, max(3, len(comparison) * .28)), layout="constrained")
    comparison.plot.barh(ax=ax, legend=False, color="#0072B2")
    ax.axvline(0, color="black", linewidth=.8)
    ax.set(title=f"Validation per-game CRPS | {report['runMode']} | negative favors B2",
           xlabel="B2 - B1 (ms)", ylabel="Game ID")
    plt.show()
    plt.close(fig)


In [ ]:
show_results(REPORT)


# 8. 결론과 다음 gate

Smoke는 구현 동작만 확인합니다. Full Validation은 별도 해석하고,
후보 ID와 bundle SHA를 고정한 다음 Test 평가 승인을 받습니다.
Test 전체·지정 게임 제외·단독은 같은 고정 모델로 fit 0회 비교합니다. 지금은 실행하지 않습니다.

0ms의 과거 원인·개인 습관·입력 성공·Hold·현재 40행 규칙·강한 gameplay AI는 미검증입니다.
Notebook을 끝내도 새 모델·수집·PVE 연결을 자동 시작하지 않습니다.

기존 full report를 다시 보려면 실행 셀 대신 JSON을 REPORT에 읽어 표시 셀을 실행합니다.
Source의 output/count는 비우고 실행 결과는 별도 executed Notebook으로 저장합니다.
